In [1]:
from itertools import product

def generar_combinaciones(elem_1,elem_2,elem_3,elem_4)->list:
    
    # combinations(lista, tamaño) devuelve un iterador
    comb = list(product(elem_1,elem_2,elem_3,elem_4))
    
    # Convertimos el iterador a lista
    return list(comb)

In [ ]:
from supervivencia import Con_elitismo, Sin_elitismo, Supervivencia
from carga import Carga
from fitness import Fitness
import numpy as np
import numpy.typing as npt
from mutacion import Mutacion, Seleccion_aleatoria, Tasa_mutacion
from cruze import Cruce_cargas, Cruce_uniforme, Cruce_un_punto, Cruce_dos_punto
from contenedor import Contenedor
from diversidad import Diversidad
from guardar import Guardar, DatosGeneraciones, AnalisisEstadistico, Configuracion, InformacionCargas

def evolucion(penalizacion_opcion: str = "fitnes_cero",
              tamano_poblacion: int = 100,
              max_generaciones: int = 100, 
              recombinacion_opcion:Cruce_cargas = Cruce_dos_punto,
              mutacion_opcion:Mutacion = Seleccion_aleatoria,
              supervivencia_opcion: Supervivencia = Con_elitismo,
              numero_ejecucion: int = 0):
    
    
    # Inicialización
    carga: Carga = Carga()
    cargas: list[npt.NDArray[np.int32]] = carga.generar_carga()

    mejor_generacion_anterior: Contenedor = None

    supervivencia: Supervivencia = supervivencia_opcion()
    fitness: Fitness 
    diversidad: Diversidad = Diversidad()
    poblacion: list[Contenedor] = []

    aptitudes:list[int] = []

    # Cargamos carga y fitness inicial a los contenedores
    
    # Llevamos la carga inicial a los contenedores
    for carga_individual in cargas:
        contenedor = Contenedor(carga=carga_individual)
        poblacion.append(contenedor)

    # Calculamos su fitnes
    for contenedor in poblacion:
        fitness = Fitness(carga=contenedor.carga)
        contenedor.fitness = fitness.calcular_fitness(penalizacion_opcion)

    # Evolución
    for generacion in range(max_generaciones):
        
        # Lo usamos para elegir el menor carga si estamos en la comfiguración elitismo.
        if not mejor_generacion_anterior == None:
            posicion, carga_minima = min(enumerate(poblacion),
                                            key=lambda x: x[1].fitness)
            
            poblacion[posicion] = mejor_generacion_anterior


        nueva_generacion, mejor_generacion_anterior = supervivencia.seleccion_supervivencia( poblacion=poblacion,
                                                                            tamano_poblacion=tamano_poblacion,
                                                                            generacion=generacion,
                                                                            max_generaciones=max_generaciones,
                                                                            mutacion=mutacion_opcion,
                                                                            recombinacion=recombinacion_opcion, 
                                                                        )
        
        poblacion = nueva_generacion

        # Calculamos su fitnes a la nueva problación
        for contenedor in poblacion:
            fitness = Fitness(carga=contenedor.carga)
            contenedor.fitness = fitness.calcular_fitness(penalizacion_opcion)

        numero_ejecucion += 1
        
    
    aptitudes = [contenedor.fitness for contenedor in poblacion]

    posicon_valor_maximo = np.argmax(aptitudes)
    posicon_valor_minimo = np.argmin(aptitudes)


    fitness_maximo = poblacion[posicon_valor_maximo].fitness
    fitness_minimo = poblacion[posicon_valor_minimo].fitness
    
    valor_promedio: float = (np.mean(aptitudes))
    valor_desviacion_std: float = (np.std(aptitudes))
    valor_varianza: float = (np.var(aptitudes))
    valor_divercidad:float = diversidad.calcular_diversidad(poblacion=poblacion)
    
    carga_fitness_maximo:Contenedor = poblacion[posicon_valor_maximo]
    carga_fitness_minimo:Contenedor = poblacion[posicon_valor_minimo]

    
    lista_cargas_minima: list[tuple[int,int,list[str],list[int]]] = []
    lista_cargas_maxima: list[tuple[int,int,list[str],list[int]]] = []
    
    (valor,peso,lista_productos, vector_carga) = carga_fitness_maximo.calcular_valor_peso()
    lista_cargas_maxima.append((valor,peso,lista_productos,vector_carga))
    
    (valor,peso,lista_productos,vector_carga) = carga_fitness_minimo.calcular_valor_peso()
    lista_cargas_minima.append((valor,peso,lista_productos,vector_carga))


    print("Número de individuos únicos (carga - diversidad genetica): ",len(set(tuple(c.carga.tolist()) for c in poblacion)))
    print("Número aptitudes únicas: (Diversidad del fitnes)", len(set(aptitudes)))
    print("Varianza:", valor_varianza)
    print("Ejemplo aptitudes:", aptitudes[:10])

    
    print(recombinacion_opcion, mutacion_opcion, penalizacion_opcion, supervivencia_opcion)


    return (fitness_maximo, fitness_minimo, valor_promedio, 
            valor_desviacion_std, valor_varianza, valor_divercidad, 
            lista_cargas_minima,
            lista_cargas_maxima,
            numero_ejecucion)
    

In [6]:

# Opciones de calculo del fitnes para la penalización.
# - fitnes_cero
# - lineal
# - cuadratica
# - exponencial


# opciones para la Mutación y la recombinacion (Cruza)
# recombinacion (Cruce_cargas)
# - Cruce_uniforme
# - Cruce_un_punto
# - Cruce_dos_punto

# mutacion (Mutacion)
# - Seleccion_aleatoria
# - Tasa_mutacion
# -- Registros --
# Métricas por generación
valor_fitness_maximo_generacion: list[int] = []
valor_fitness_minimo_generacion: list[int] = []
valor_promedio_generacion: list[float] = []
valor_desviacion_std_generacion: list[float] = []
valor_varianza_generacion: list[float] = []
valor_diversidad_generacion: list[float] = []


lista_configuraciones: list[list[Cruce_cargas,Mutacion,str,Supervivencia,]] = []
lista_configuraciones_totales: list[list[tuple[int,float,float,float,Cruce_cargas,Mutacion,str,Supervivencia]]] = []

# --- Configuraciones ---
recombinacion_opcion: list[Cruce_cargas] = [Cruce_uniforme, Cruce_un_punto, Cruce_dos_punto]
mutacion_opcion: list[Mutacion]= [Seleccion_aleatoria,Tasa_mutacion]
penalizacion_opcion: list[str] = ['fitnes_cero','lineal','cuadratica','exponencial']
supervivencia_opcion: list[Supervivencia] = [Con_elitismo,Sin_elitismo]

lista_informacion_cargas_maxima: list[list[tuple[int,int,list[str]]]] = []
lista_informacion_cargas_minima: list[list[tuple[int,int,list[str]]]] = []

configuraciones_posibles:list[list[Cruce_cargas,Mutacion,str,Supervivencia]] = generar_combinaciones(recombinacion_opcion,mutacion_opcion,penalizacion_opcion,supervivencia_opcion)

num_ejecuciones:int = 0
for posicion,(recombinacion_opcion,mutacion_opcion,penalizacion_opcion,supervivencia_opcion) in enumerate(configuraciones_posibles):
    
    lista_configuraciones.append([recombinacion_opcion,mutacion_opcion,penalizacion_opcion,supervivencia_opcion])
     
    (fitness_maximo, 
     fitness_minimo, 
     valor_promedio, 
     valor_desviacion_std, 
     valor_varianza,
     valor_divercidad,
     informacion_cargas_min,
     informacion_cargas_max,
     numero_ejecucion)=evolucion(penalizacion_opcion=penalizacion_opcion,
                                 recombinacion_opcion=recombinacion_opcion,
                                 mutacion_opcion=mutacion_opcion,
                                 supervivencia_opcion=supervivencia_opcion,
                                 numero_ejecucion=num_ejecuciones)

    valor_fitness_maximo_generacion.append(fitness_maximo)
    valor_fitness_minimo_generacion.append(fitness_minimo)
    valor_promedio_generacion.append(valor_promedio)
    valor_desviacion_std_generacion.append(valor_desviacion_std)
    valor_varianza_generacion.append(valor_varianza)
    valor_diversidad_generacion.append(valor_divercidad)
    
    lista_informacion_cargas_maxima.append(informacion_cargas_max)
    lista_informacion_cargas_minima.append(informacion_cargas_min)

    lista_configuraciones_totales.append([fitness_maximo,valor_promedio,
                                          valor_varianza,valor_desviacion_std,
                                          recombinacion_opcion,mutacion_opcion,
                                          penalizacion_opcion,supervivencia_opcion])

    num_ejecuciones = numero_ejecucion


    #print(num_ejecuciones)

#Imprimimos el numero de ejecuciones:
print(f"Número de ejecuciones {num_ejecuciones}")

# Guardamos los datos de la configuración en archivos CSV
guardar_configuracion: Guardar 
guardar_datos_generaciones: Guardar
guardar_informacion_cargas_max:Guardar
guardar_informacion_cargas_min:Guardar
guardar_analisis_estadistico:Guardar

guardar_configuracion = Configuracion(configuraciones_posibles=configuraciones_posibles)

guardar_datos_generaciones = DatosGeneraciones(valor_varianza_generacion=valor_diversidad_generacion,
                           valor_diversidad_generacion=valor_diversidad_generacion,
                           valor_desviacion_std_generacion=valor_desviacion_std_generacion,
                           valor_fitness_maximo_generacion=valor_fitness_maximo_generacion,
                           valor_fitness_minimo_generacion=valor_fitness_minimo_generacion,
                           valor_promedio_generacion=valor_promedio_generacion)


guardar_informacion_cargas_max = InformacionCargas(lista_informacion_cargas=lista_informacion_cargas_maxima,
                                               nombre_archivo="datos_carga_maxima")
guardar_informacion_cargas_min = InformacionCargas(lista_informacion_cargas=lista_informacion_cargas_minima,
                                               nombre_archivo="datos_carga_minima")
guardar_analisis_estadistico = AnalisisEstadistico(lista_configuraciones_totales=lista_configuraciones_totales)

guardar_configuracion.guardar_CSV()
guardar_datos_generaciones.guardar_CSV()
guardar_informacion_cargas_max.guardar_CSV()
guardar_informacion_cargas_min.guardar_CSV()
guardar_analisis_estadistico.guardar_CSV()

12
Varianza: 7393.4411
Valores únicos: 6
Ejemplo aptitudes: [301, 301, 301, 301, 301, 301, 301, 301, 301, 301]
<class 'cruze.cruce_uniforme.Cruce_uniforme'> <class 'mutacion.seleccion_aleatoria.Seleccion_aleatoria'> fitnes_cero <class 'supervivencia.con_elitismo.Con_elitismo'>
15
Varianza: 6248.229600000001
Valores únicos: 8
Ejemplo aptitudes: [292, 292, 292, 292, 292, 292, 292, 292, 292, 292]
<class 'cruze.cruce_uniforme.Cruce_uniforme'> <class 'mutacion.seleccion_aleatoria.Seleccion_aleatoria'> fitnes_cero <class 'supervivencia.sin_elitismo.Sin_elitismo'>
16
Varianza: 111.7299
Valores únicos: 14
Ejemplo aptitudes: [912, 912, 912, 912, 912, 912, 860, 912, 912, 876]
<class 'cruze.cruce_uniforme.Cruce_uniforme'> <class 'mutacion.seleccion_aleatoria.Seleccion_aleatoria'> lineal <class 'supervivencia.con_elitismo.Con_elitismo'>
9
Varianza: 124.54000000000002
Valores únicos: 8
Ejemplo aptitudes: [912, 912, 912, 902, 912, 912, 912, 912, 912, 912]
<class 'cruze.cruce_uniforme.Cruce_uniforme'

In [ ]:
# Graficar
from grafica import Grafica, Grafica_lineas, Grafica_lineas_dobles

    # grafica_promedio: Grafica = Grafica_lineas()
    # grafica_maximo: Grafica = Grafica_lineas()
    # grafica_minimo: Grafica = Grafica_lineas()
    # grafica_desviacion_std: Grafica = Grafica_lineas()
#grafica_mejora_generacion: Grafica = Grafica_lineas()
    # grafica_varianza: Grafica = Grafica_lineas()
    # grafica_diversidad: Grafica = Grafica_lineas()

    # grafica_promedio.generar_grafica(datos=valor_promedio_generacion,
    #                                  etiqueta="Promedio",
    #                                  guardar_grafica=("Promedio_"+configuracion_opcion))

    # grafica_maximo.generar_grafica(datos=valor_maximo_generacion,
    #                                etiqueta="Maximo",
    #                                guardar_grafica=("Maximo_"+configuracion_opcion))

    # grafica_minimo.generar_grafica(datos=valor_minimo_generacion,
    #                                etiqueta="Minimo",
    #                                guardar_grafica=("Minio_"+configuracion_opcion))

    # grafica_desviacion_std.generar_grafica(datos=valor_desviacion_std_generacion,
    #                                        etiqueta="Desviación estándar",
    #                                        guardar_grafica=("DesviacionEstandar_"+configuracion_opcion))

    # grafica_varianza.generar_grafica(datos=valor_varianza_generacion,
    #                                  etiqueta=" Varianza",
    #                                  guardar_grafica=("Varianza_"+configuracion_opcion))

    # grafica_diversidad.generar_grafica(datos=valor_diversidad_generacion,
    #                                    etiqueta=" Diversidad",
    #                                    guardar_grafica=("Diversidad_"+configuracion_opcion))

    # mejora = np.diff(valor_maximo_generacion)

    # grafica_mejora_generacion.generar_grafica(datos=mejora,
    #                                           etiqueta="Mejora entre cada generación",
    #                                           guardar_grafica=("MejoraGeneraciones_"+configuracion_opcion))

In [7]:
from contenedor import Contenedor
from fitness import Fitness
def calcular_pesos(contenedor: Contenedor):
    fitness:Fitness = Fitness(carga=contenedor.carga)
    contenedor.fitness = fitness.calcular_fitness()
    #valor_carga, peso_carga, lista_productos, self.carga 
    valor, peso, lista_productos, vertor_carga = contenedor.calcular_valor_peso()
    print(f"Valor  {valor}")
    print(f"Peso {peso}")
    print(f"Lista de productos {lista_productos}")
    print(f"Vector de carga {vertor_carga}")

carga = [0,0,0,1,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,1,1,0,0,0]
cg:Contenedor = Contenedor(carga= carga)
calcular_pesos(contenedor=cg)

     
    

Valor  347
Peso 50
Lista de productos ['Linterna LED', 'Baterı́as recargables', 'Cuchillo multiusos', 'Filtro portátil de agua', 'Cartucho de gas', 'GPS de montaña', 'Teléfono satelital', 'Laptop ultraligera', 'Power bank alta capacidad']
Vector de carga [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0]
